# 实验 8 — dlt CLI：pipeline show / trace

**目标**：探索 dlt 的命令行工具，理解 pipeline state 和 load trace 的含义。

In [ ]:
import subprocess

def dlt_cli(args: list[str]) -> str:
    result = subprocess.run(
        ["uv", "run", "dlt"] + args,
        capture_output=True, text=True, cwd=".."
    )
    return result.stdout + result.stderr

print("=== dlt pipeline list ===")
print(dlt_cli(["pipeline", "list"]))

In [ ]:
print("=== dlt pipeline ecb_rates_pipeline info ===")
print(dlt_cli(["pipeline", "ecb_rates_pipeline", "info"]))

In [ ]:
print("=== dlt pipeline ecb_rates_pipeline trace ===")
print(dlt_cli(["pipeline", "ecb_rates_pipeline", "trace"]))

In [ ]:
# last_trace 也可以在 Python 里直接访问
import dlt

pipeline = dlt.pipeline(
    pipeline_name="ecb_rates_pipeline",
    destination=dlt.destinations.duckdb("../data/sandbox.duckdb"),
    dataset_name="raw_ecb",
)
print("last_trace:")
print(pipeline.last_trace)

In [ ]:
# pipeline state 存在哪？
import duckdb

con = duckdb.connect("../data/sandbox.duckdb")
print("=== _dlt_pipeline_state ===")
con.execute("SELECT * FROM raw_ecb._dlt_pipeline_state ORDER BY created_at DESC LIMIT 3").df()

## 思考

- `dlt pipeline show` 会启动一个本地 web UI（需要交互式终端），在终端里运行：
  ```bash
  uv run dlt pipeline ecb_rates_pipeline show
  ```
- `_dlt_pipeline_state` 存储 pipeline 的状态（如增量游标），下次运行时 dlt 可以从上次中断的地方恢复
- `last_trace` 包含每个步骤（extract/normalize/load）的耗时和行数统计